# 02 - Classification

Train, compare and explain linear text classifiers with `newsroom.classification`.

The pipeline is `TextCleaner -> FeatureUnion(word TF-IDF, char TF-IDF) -> classifier`, optionally wrapped in `CalibratedClassifierCV` so every model exposes probabilities.

In [ ]:
import pandas as pd

from newsroom.data.corpus import generate_corpus
from newsroom.classification.trainer import TrainConfig, train_classifier

df = generate_corpus(n_docs=2000, seed=42)
df["label"].value_counts()

## Compare the supported models

All four models share the same features; only the estimator changes.

In [ ]:
rows = []
results = {}
for model in ("linear_svc", "logistic_regression", "complement_nb", "sgd"):
    result = train_classifier(df, TrainConfig(model=model, grid_search=False))
    results[model] = result
    m = result.metrics
    rows.append({"model": model, "accuracy": m.accuracy, "macro_f1": m.macro_f1, "seconds": result.train_seconds})
pd.DataFrame(rows).round(4)

## Grid search

A compact grid over n-gram range and regularisation strength, scored by macro-F1 with stratified folds.

In [ ]:
best = train_classifier(df, TrainConfig(model="linear_svc", grid_search=True, cv_folds=3))
print(best.best_params)
print(f"accuracy={best.metrics.accuracy:.4f} macro_f1={best.metrics.macro_f1:.4f}")

## Per-class report and confusion matrix

In [ ]:
from IPython.display import Image

from newsroom.evaluation.metrics import classification_report_text
from newsroom.visualization.plots import plot_confusion_matrix

print(classification_report_text(best.metrics))
Image(filename=str(plot_confusion_matrix(best.metrics.confusion, best.metrics.labels, "../artifacts/notebooks/confusion.png")))

## What did the model learn?

Linear coefficients give a direct view of the most indicative terms per class, and `explain_document` attributes an individual prediction to its features.

In [ ]:
from newsroom.classification.explain import explain_document, top_terms_per_class
from newsroom.visualization.plots import plot_top_terms

terms = top_terms_per_class(best.pipeline, n_terms=8)
for label, pairs in terms.items():
    print(f"{label:<11}", [t for t, _ in pairs])
Image(filename=str(plot_top_terms(terms, "../artifacts/notebooks/top_terms.png")))

In [ ]:
from newsroom.classification.predictor import Predictor

predictor = Predictor(best.pipeline)
text = "The goalkeeper saved a late penalty as the home side reached the cup final."
print(predictor.predict(text).top(3))
explain_document(best.pipeline, text, n_terms=6)

## Save for the API

`save_model` writes a joblib artifact that `newsroom serve` loads at startup.

In [ ]:
from newsroom.classification.trainer import save_model

save_model(best, "../artifacts/classifier.joblib")